In [1]:
import socket
print(socket.gethostname())

awr-1-54


In [2]:
import os
import xesmf as xe
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt

In [3]:
from dask.distributed import Client
client = Client(n_workers=8, threads_per_worker=4)

print(client.dashboard_link)
print(client)

http://127.0.0.1:8787/status
<Client: 'tcp://127.0.0.1:39435' processes=8 threads=32, memory=64.00 GiB>


### Anemoi Inference
Inference has been done for two different datasets:
- Terrain-Following Coordinate System (TFCS) Dataset
- Pressure-Level Coordinate System (PLCS) Dataset

TFCS dataset has been resampled by xarray resampling method, while PLCS has been subsampled by atmospheric conventional practice. Thus, the initial condition for each dataset should be determined differently.

- **xarray dataset** (`T00` = avg of 0–6, i.e. start-labeled): the block that **starts** at your target time — so the label is just the target time itself, unchanged.
- **Atmospheric-convention dataset** (`T06` = avg of 0–6, i.e. end-labeled): the same block is labeled by its **end** time — so the label is target time **+ 6 hours**.


| Original | −48h (raw) | Target IC time | xarray label (as-is) | Atmospheric label (+6h) |
|---|---|---|---|---|
| 2022-12-27-T10:00:00 | 2022-12-25 10:00 | **2022-12-25-T12:00:00**                   | 2022-12-25-T12:00:00 | 2022-12-25-T18:00:00 | 
| 2022-12-30-T21:00:00 | 2022-12-28 21:00 | **2022-12-29-T00:00:00** (tie, rounded up) | 2022-12-29-T00:00:00 | 2022-12-29-T06:00:00 |
| 2023-01-05-T01:00:00 | 2023-01-03 01:00 | **2023-01-03-T00:00:00**                   | 2023-01-03-T00:00:00 | 2023-01-03-T06:00:00 |
| 2023-01-08-T07:00:00 | 2023-01-06 07:00 | **2023-01-06-T06:00:00**                   | 2023-01-06-T06:00:00 | 2023-01-06-T12:00:00 |
| 2023-01-09-T09:00:00 | 2023-01-07 09:00 | **2023-01-07-T12:00:00** (tie, rounded up) | 2023-01-07-T12:00:00 | 2023-01-07-T18:00:00 |
| 2023-01-13-T16:00:00 | 2023-01-11 16:00 | **2023-01-11-T18:00:00**                   | 2023-01-11-T18:00:00 | 2023-01-12-T00:00:00 |
| 2023-03-10-T09:00:00 | 2023-03-08 09:00 | **2023-03-08-T12:00:00** (tie, rounded up) | 2023-03-08-T12:00:00 | 2023-03-08-T18:00:00 |
| 2023-03-14-T15:00:00 | 2023-03-12 15:00 | **2023-03-12-T18:00:00** (tie, rounded up) | 2023-03-12-T18:00:00 | 2023-03-13-T00:00:00 |

In [ ]:
ds_tfcs_gt = xr.open_dataset('/cw3e/mead/projects/cwp167/moerfani_data/anemoi-output/ar-analysis/TFCS-GT/2022-12-25.nc')
ds_tfcs_pd = xr.open_dataset('/cw3e/mead/projects/cwp167/moerfani_data/anemoi-output/ar-analysis/TFCS/2022-12-25.nc')

In [5]:
def regrid_data(ds, in_grid, out_grid, exclude_vars):
    """
    Regrid the data in the given xarray Dataset from the input grid to the output grid.

    Parameters:
    - ds: xarray Dataset containing the data to be regridded.
    - in_grid: Dictionary defining the input grid.
    - out_grid: Dictionary defining the output grid.
    - exclude_vars: List of variable names to exclude from regridding.

    Returns:
    - out_arrays: Dictionary containing regridded arrays for each variable.
    - output_grid_info: Information about the output grid.
    """
    from earthkit.geo.grids.array import regrid

    # --------------------------------------------------
    # Store results
    # --------------------------------------------------

    out_arrays = {}
    output_grid_info = None


    # --------------------------------------------------
    # Regrid all variables
    # --------------------------------------------------

    for var in ds.data_vars:

        # Skip original N320 latitude/longitude arrays
        if var in exclude_vars:
            continue

        print(f"\nProcessing: {var}")

        # Conservative interpolation for precipitation
        if var == "tp":
            interpolation = "grid-box-average"
        else:
            interpolation = "linear"

        print(f"Interpolation: {interpolation}")

        results = []

        # --------------------------------------------------
        # Loop through time
        # --------------------------------------------------

        for t in range(ds.sizes["time"]):

            # Extract one N320 field
            # Shape should be (542080,)
            vals = ds[var].isel(time=t).values

            if vals.shape != (542080,):
                raise ValueError(
                    f"{var}, time index {t}: "
                    f"expected shape (542080,), got {vals.shape}"
                )

            # --------------------------------------------------
            # Earthkit-Geo regridding
            # --------------------------------------------------

            result, result_grid = regrid(
                vals,
                in_grid=in_grid,
                out_grid=out_grid,
                interpolation=interpolation,
                backend="mir",
            )

            results.append(result)

            # Save normalized output grid information once
            if output_grid_info is None:
                output_grid_info = result_grid

        # Stack time dimension back together
        out_arrays[var] = np.stack(results)

        print("Input shape: ", ds[var].shape)
        print("Output shape:", out_arrays[var].shape)


    print("\nOutput grid:")
    print(output_grid_info)

    return out_arrays, output_grid_info

In [9]:
# --------------------------------------------------
# Grid definitions
# --------------------------------------------------

in_grid = {"grid": "N320"}
out_grid = {"grid": [0.25, 0.25]}


# --------------------------------------------------
# Variables that should NOT be regridded as fields
# --------------------------------------------------

exclude_vars = [
    "latitude",
    "longitude",
    "sin_julian_day",
    "sin_latitude",
    "sin_local_time",
    "sin_longitude",
    "cos_julian_day",
    "cos_latitude",
    "cos_local_time",
    "cos_longitude",
    "insolation",
    "lsm",
    "tsir"
]

out_arr_tfcs_gt, output_grid_info = regrid_data(ds_tfcs_gt, in_grid, out_grid, exclude_vars)
out_arr_tfcs_pd, _ = regrid_data(ds_tfcs_pd, in_grid, out_grid, exclude_vars)


Processing: 10u
Interpolation: linear
Input shape:  (13, 542080)
Output shape: (13, 721, 1440)

Processing: 10v
Interpolation: linear
Input shape:  (13, 542080)
Output shape: (13, 721, 1440)

Processing: 2t
Interpolation: linear
Input shape:  (13, 542080)
Output shape: (13, 721, 1440)

Processing: d2m
Interpolation: linear
Input shape:  (13, 542080)
Output shape: (13, 721, 1440)

Processing: ivt_u
Interpolation: linear
Input shape:  (13, 542080)
Output shape: (13, 721, 1440)

Processing: ivt_v
Interpolation: linear
Input shape:  (13, 542080)
Output shape: (13, 721, 1440)

Processing: iwv
Interpolation: linear
Input shape:  (13, 542080)
Output shape: (13, 721, 1440)

Processing: msl
Interpolation: linear
Input shape:  (13, 542080)
Output shape: (13, 721, 1440)

Processing: q_100
Interpolation: linear
Input shape:  (13, 542080)
Output shape: (13, 721, 1440)

Processing: q_1000
Interpolation: linear
Input shape:  (13, 542080)
Output shape: (13, 721, 1440)

Processing: q_150
Interpolation

In [10]:
def create_output_dataset(out_arrays, ds_anemoi):
    """
    Create an xarray Dataset from the regridded arrays.

    Parameters:
    - out_arrays: Dictionary containing regridded arrays for each variable.
    - ds_anemoi: Original xarray Dataset containing time information.

    Returns:
    - ds_out: xarray Dataset containing the regridded data and metadata.
    """
    # --------------------------------------------------
    # Output grid
    # --------------------------------------------------

    nlat = 721
    nlon = 1440

    # Earthkit output for this grid is north -> south
    lat = np.linspace(90.0, -90.0, nlat)

    # 0, 0.25, ..., 359.75
    lon = np.arange(0.0, 360.0, 0.25)

    time = ds_anemoi["time"].values


    # --------------------------------------------------
    # Pressure levels
    # --------------------------------------------------

    pressure_levels = np.array(
        [50, 100, 150, 200, 250, 
        300, 400, 500, 550, 600, 
        650, 700, 750, 800, 850, 
        900, 925, 950, 1000
        ]
        )

    pressure_vars = ["q", "t", "u", "v", "z"]


    # --------------------------------------------------
    # Create Dataset
    # --------------------------------------------------

    data_vars = {}


    # --------------------------------------------------
    # Pressure-level variables
    # --------------------------------------------------

    for var in pressure_vars:

        arrays = []

        for level in pressure_levels:

            name = f"{var}_{level}"

            if name not in out_arrays:
                raise KeyError(f"{name} not found in out_arrays")

            arrays.append(out_arrays[name])

        # (level, time, lat, lon)
        stacked = np.stack(arrays, axis=0)

        # Reorder to:
        # (time, level, latitude, longitude)
        stacked = np.transpose(stacked, (1, 0, 2, 3))

        data_vars[var] = (
            ("time", "level", "lat", "lon"),
            stacked,
        )


    # --------------------------------------------------
    # Surface / 2D variables
    # --------------------------------------------------

    for var, arr in out_arrays.items():

        # Pressure-level variables have already been handled
        if any(var == f"{p}_{level}"
            for p in pressure_vars
            for level in pressure_levels):
            continue

        data_vars[var] = (
            ("time", "lat", "lon"),
            arr,
        )


    # --------------------------------------------------
    # Create final Dataset
    # --------------------------------------------------

    ds_out = xr.Dataset(
        data_vars=data_vars,
        coords={
            "time": time,
            "level": pressure_levels,
            "lat": lat,
            "lon": lon,
        },
    )


    # --------------------------------------------------
    # Add useful metadata
    # --------------------------------------------------

    ds_out["level"].attrs = {
        "long_name": "pressure level",
        "units": "hPa",
    }

    ds_out["lat"].attrs = {
        "standard_name": "latitude",
        "units": "degrees_north",
    }

    ds_out["lon"].attrs = {
        "standard_name": "longitude",
        "units": "degrees_east",
    }

    ds_out['ivt'] = np.sqrt(ds_out["ivt_u"]**2 + ds_out["ivt_v"]**2)

    return ds_out


In [ ]:
ds_tfcs_gt_025 = create_output_dataset(out_arr_tfcs_gt, ds_tfcs_gt)
ds_tfcs_pd_025 = create_output_dataset(out_arr_tfcs_pd, ds_tfcs_pd)

In [14]:
def calculate_pressure_field(ds_out):
    """
    Calculate the pressure field at model levels using ECMWF hybrid coefficients.

    Parameters:
    - ds_out: xarray Dataset containing surface pressure and other variables.

    Returns:
    - ds_out: xarray Dataset with the calculated pressure field added.
    """

    import pandas as pd

    # Read the ECMWF coefficients (alpha and beta)
    pressure_levels = np.array([50, 100, 150, 200, 250, 300, 400, 500, 550, 600, 650, 700, 750, 800, 850, 900, 925, 950, 1000])
    vertical_subset = [48, 60, 68, 74, 79, 83, 90, 96, 98, 101, 103, 105, 108, 111, 114, 118, 120, 123, 133]

    df = pd.read_csv('ecmwf_coeffs.csv')

    # 1. Extract the 138 Half-Level Coefficients
    a_half = df['alpha'].to_numpy() / 100.0  
    b_half = df['beta'].to_numpy()

    # 2. Calculate the 137 Full-Level Coefficients (Mass Levels)
    # We slice the arrays to average adjacent values: (top_boundary + bottom_boundary) / 2
    a_full = (a_half[:-1] + a_half[1:]) / 2.0
    b_full = (b_half[:-1] + b_half[1:]) / 2.0

    a_full = a_full[vertical_subset]
    b_full = b_full[vertical_subset]

    # 3. Convert coefficients into xarray DataArrays
    A_full = xr.DataArray(a_full, dims=["level"], coords={"level": pressure_levels}).astype(np.float32)
    B_full = xr.DataArray(b_full, dims=["level"], coords={"level": pressure_levels}).astype(np.float32)

    # 4. Calculate the Final 3D/4D Target Pressure Field
    p_sfc_hPa = ds_out['sp'].astype(np.float32) / 100.0

    p_field = A_full + B_full * p_sfc_hPa

    ds_out['p'] = p_field.transpose('time', 'level', 'lat', 'lon')
    ds_out['p'].attrs = {
        'long_name': 'Pressure at model levels',
        'units': 'hPa',
        'description': 'Calculated pressure at model levels using ECMWF hybrid coefficients',
    }

    return ds_out

In [ ]:
ds_tfcs_gt_025 = calculate_pressure_field(ds_tfcs_gt_025)
ds_tfcs_pd_025 = calculate_pressure_field(ds_tfcs_pd_025)

In [18]:
ds_tfcs_gt_025.to_netcdf('/cw3e/mead/projects/cwp167/moerfani_data/anemoi-output/ar-analysis/TFCS-GT/2022-12-25_025.nc')
ds_tfcs_pd_025.to_netcdf('/cw3e/mead/projects/cwp167/moerfani_data/anemoi-output/ar-analysis/TFCS/2022-12-25_025.nc')